In [ ]:
# Step 1 - Import packages

from __future__ import annotations

import os
import re
import shutil
import tempfile
import warnings
import zipfile
from io import BytesIO
from pathlib import Path, PurePosixPath

import numpy as np
import pandas as pd
from openpyxl import load_workbook
from openpyxl.styles import Alignment, Font, PatternFill
from openpyxl.utils import get_column_letter

try:
    from IPython.display import display
except ImportError:
    def display(value):
        print(value)

pd.set_option("display.max_columns", 80)  # Show enough columns during schema checks.
pd.set_option("display.max_rows", 100)
warnings.filterwarnings("once", category=UserWarning)

print("pandas:", pd.__version__)

In [ ]:
# Step 2 - Set paths and extraction rules

NOTEBOOK_NAME = "transition-data"

# On the remote computer this resolves to Z:\Research-analysis.
RESEARCH_ROOT = Path(os.environ.get("RESEARCH_ANALYSIS_ROOT", r"Z:\Research-analysis"))
RAW_ZIP_PATH = Path(
    os.environ.get(
        "LUTO_TRANSFER_ZIP",
        str(RESEARCH_ROOT / "data" / "raw" / "LUTO2-Scenario-Run-2" / "transfer_3573951_files_89a7061e.zip"),
    )
)
FILE_INVENTORY_PATH = Path(
    os.environ.get(
        "LUTO_FILE_INVENTORY",
        str(RESEARCH_ROOT / "0-Notes" / "Combined_file_inventory.xlsx"),
    )
)

OUTPUT_DIR = RESEARCH_ROOT / "data" / "processed" / NOTEBOOK_NAME
EXCEL_PATH = OUTPUT_DIR / f"{NOTEBOOK_NAME}.xlsx"
VALIDATION_CSV_PATH = OUTPUT_DIR / f"{NOTEBOOK_NAME}_validation.csv"

SELECTED_RUNS = list(range(1, 17))
EXPECTED_YEARS = [2025, 2030, 2035, 2040, 2045, 2050]
FAMILY_SPECS = {
    "transition_ag2ag_area": {
        "transition_direction": "Agriculture to Agriculture", "measure": "Area", "unit": "ha",
        "sheet_prefix": "Ag2Ag_area",
    },
    "transition_ag2ag_cost": {
        "transition_direction": "Agriculture to Agriculture", "measure": "Cost", "unit": "AUD",
        "sheet_prefix": "Ag2Ag_cost",
    },
    "transition_ag2nonag_area": {
        "transition_direction": "Agriculture to Non-Agriculture", "measure": "Area", "unit": "ha",
        "sheet_prefix": "Ag2NonAg_area",
    },
    "transition_ag2nonag_cost": {
        "transition_direction": "Agriculture to Non-Agriculture", "measure": "Cost", "unit": "AUD",
        "sheet_prefix": "Ag2NonAg_cost",
    },
    "transition_nonag2ag_area": {
        "transition_direction": "Non-Agriculture to Agriculture", "measure": "Area", "unit": "ha",
        "sheet_prefix": "NonAg2Ag_area",
    },
    "transition_nonag2ag_cost": {
        "transition_direction": "Non-Agriculture to Agriculture", "measure": "Cost", "unit": "AUD",
        "sheet_prefix": "NonAg2Ag_cost",
    },
}
EXPECTED_FILE_COUNT = len(SELECTED_RUNS) * len(EXPECTED_YEARS) * len(FAMILY_SPECS)
EXPECTED_NATIONAL_ROWS = EXPECTED_FILE_COUNT
EXPECTED_RUN_YEAR_ROWS = len(SELECTED_RUNS) * len(EXPECTED_YEARS)
EXCEL_DATA_ROW_LIMIT = 1_048_575  # One row is reserved for the Excel header.

# F: is intended for temporary large files on the remote Windows computer.
preferred_temp_root = Path(os.environ.get("LUTO_TEMP_ROOT", r"F:\transition-data-temp"))

print("Source ZIP:", RAW_ZIP_PATH)
print("Inventory:", FILE_INVENTORY_PATH)
print("Final workbook:", EXCEL_PATH)

In [ ]:
# Step 3 - Pre-validation

assert RAW_ZIP_PATH.is_file(), f"Source ZIP not found: {RAW_ZIP_PATH}"
assert FILE_INVENTORY_PATH.is_file(), f"Inventory workbook not found: {FILE_INVENTORY_PATH}"

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)  # Create the requested output folder.

# Use F: when it exists; otherwise use the operating system's temporary directory.
if preferred_temp_root.drive and Path(preferred_temp_root.anchor).exists():
    TEMP_ROOT = preferred_temp_root
else:
    TEMP_ROOT = Path(tempfile.gettempdir()) / "transition-data-temp"
TEMP_ROOT.mkdir(parents=True, exist_ok=True)

print("Preflight passed.")
print("Temporary extraction folder:", TEMP_ROOT)
print("Temporary free space (GB):", round(shutil.disk_usage(TEMP_ROOT).free / 1024**3, 2))

In [ ]:
# Step 4 - Identify exact source files from the inventory

inventory_parts = []

for run_id in SELECTED_RUNS:
    sheet_name = f"Run_G{run_id:04d}"
    part = pd.read_excel(FILE_INVENTORY_PATH, sheet_name=sheet_name)
    part["run_id"] = run_id
    part["run_code"] = sheet_name
    inventory_parts.append(part)

inventory_all = pd.concat(inventory_parts, ignore_index=True, sort=False)
inventory_all["file_name"] = inventory_all["file_name"].astype("string").str.strip()

selected_parts = []
year_pattern = "|".join(str(year) for year in EXPECTED_YEARS)

for family_key, specification in FAMILY_SPECS.items():
    pattern = re.compile(rf"^{re.escape(family_key)}_({year_pattern})\.csv$", re.IGNORECASE)
    matched = inventory_all.loc[inventory_all["file_name"].str.fullmatch(pattern, na=False)].copy()
    matched["family_key"] = family_key
    matched["transition_direction"] = specification["transition_direction"]
    matched["measure"] = specification["measure"]
    matched["unit"] = specification["unit"]
    matched["sheet_prefix"] = specification["sheet_prefix"]
    matched["source_year"] = matched["file_name"].str.extract(pattern, expand=False).astype("int64")
    selected_parts.append(matched)

selected_inventory = pd.concat(selected_parts, ignore_index=True, sort=False)
selected_inventory["file_location"] = selected_inventory["file_location"].astype("string")
selected_inventory[["inner_archive_member", "csv_member"]] = (
    selected_inventory["file_location"]
    .str.split(" :: ", n=1, expand=True)
    .rename(columns={0: "inner_archive_member", 1: "csv_member"})
)
selected_inventory = selected_inventory.sort_values(
    ["run_id", "source_year", "family_key"]
).reset_index(drop=True)

key_columns = ["run_id", "source_year", "family_key"]
duplicate_keys = selected_inventory.duplicated(key_columns, keep=False)
assert len(selected_inventory) == EXPECTED_FILE_COUNT, (
    f"Expected {EXPECTED_FILE_COUNT} files but identified {len(selected_inventory)}."
)
assert not duplicate_keys.any(), "Duplicate run/year/transition-family files were identified."
assert selected_inventory["csv_member"].notna().all(), "A nested CSV member path is missing."

coverage = pd.crosstab(
    [selected_inventory["run_id"], selected_inventory["source_year"]],
    selected_inventory["family_key"],
)
assert coverage.shape == (EXPECTED_RUN_YEAR_ROWS, len(FAMILY_SPECS))
assert coverage.eq(1).all().all(), "Inventory coverage is not exactly one file per combination."

print(f"Identified {len(selected_inventory)} source files with complete coverage.")
display(selected_inventory[["run_code", "source_year", "family_key", "file_name", "file_location"]].head(12))

In [ ]:
# Step 5 - Define ZIP readers

def _zip_name_map(zip_file: zipfile.ZipFile) -> dict[str, str]:
    # Map normalised lowercase ZIP paths to their original member names.
    return {
        str(PurePosixPath(name.replace("\\", "/"))).lower(): name
        for name in zip_file.namelist()
        if not name.endswith("/")
    }


def resolve_zip_member(zip_file: zipfile.ZipFile, requested: str) -> str:
    # Resolve a member path case-insensitively and tolerate slash differences.
    normalised = str(PurePosixPath(str(requested).replace("\\", "/"))).lower()
    name_map = _zip_name_map(zip_file)
    if normalised not in name_map:
        basename_matches = [
            original
            for key, original in name_map.items()
            if PurePosixPath(key).name == PurePosixPath(normalised).name
        ]
        if len(basename_matches) == 1:
            return basename_matches[0]
        raise KeyError(f"ZIP member not found or ambiguous: {requested}")
    return name_map[normalised]


def read_csv_from_zip(zip_file: zipfile.ZipFile, member_name: str) -> tuple[pd.DataFrame, str]:
    # Read a CSV with a short sequence of common encodings.
    resolved = resolve_zip_member(zip_file, member_name)
    raw = zip_file.read(resolved)
    last_error = None
    for encoding in ("utf-8-sig", "utf-8", "cp1252"):
        try:
            return pd.read_csv(BytesIO(raw), encoding=encoding, low_memory=False), encoding
        except UnicodeDecodeError as error:
            last_error = error
    raise last_error


def safe_sheet_token(text: str) -> str:
    # Return a short token that is safe to use in an Excel sheet name.
    return re.sub(r"[^A-Za-z0-9_]", "_", str(text))[:24]

In [ ]:
# Step 6 - Extract all source rows and record audit

raw_parts = []
extraction_audit_rows = []

with zipfile.ZipFile(RAW_ZIP_PATH, "r") as outer_zip:
    for run_id in SELECTED_RUNS:
        run_selection = selected_inventory.loc[selected_inventory["run_id"].eq(run_id)].copy()
        inner_members = run_selection["inner_archive_member"].dropna().unique().tolist()
        assert len(inner_members) == 1, f"Run {run_id} points to {len(inner_members)} nested archives."

        requested_inner = inner_members[0]
        resolved_inner = resolve_zip_member(outer_zip, requested_inner)
        inner_info = outer_zip.getinfo(resolved_inner)

        # Leave a 2 GB safety margin beyond the nested archive's uncompressed size.
        required_free = inner_info.file_size + 2 * 1024**3
        available_free = shutil.disk_usage(TEMP_ROOT).free
        if available_free < required_free:
            raise OSError(
                f"Not enough temporary free space for {resolved_inner}. "
                f"Required about {required_free / 1024**3:.1f} GB; "
                f"available {available_free / 1024**3:.1f} GB. "
                "Set LUTO_TEMP_ROOT to a drive with more free space."
            )

        print(f"Run {run_id:02d}: materialising one nested archive ({inner_info.file_size / 1024**3:.1f} GB)...")
        with tempfile.TemporaryDirectory(prefix=f"transition_run_{run_id:02d}_", dir=TEMP_ROOT) as temp_dir:
            temp_archive = Path(temp_dir) / f"Run_G{run_id:04d}_Archive.zip"

            # Stream the nested archive to disk without holding it in memory.
            with outer_zip.open(resolved_inner, "r") as source, temp_archive.open("wb") as target:
                shutil.copyfileobj(source, target, length=16 * 1024**2)

            with zipfile.ZipFile(temp_archive, "r") as run_zip:
                for record in run_selection.itertuples(index=False):
                    try:
                        frame, encoding = read_csv_from_zip(run_zip, record.csv_member)
                        frame.insert(0, "source_row", np.arange(1, len(frame) + 1, dtype="int64"))
                        frame.insert(0, "source_member", str(record.csv_member))
                        frame.insert(0, "source_file", str(record.file_name))
                        frame.insert(0, "source_year", int(record.source_year))
                        frame.insert(0, "unit", str(record.unit))
                        frame.insert(0, "measure", str(record.measure))
                        frame.insert(0, "transition_direction", str(record.transition_direction))
                        frame.insert(0, "family_key", str(record.family_key))
                        frame.insert(0, "run_code", str(record.run_code))
                        frame.insert(0, "run_id", int(record.run_id))
                        raw_parts.append(frame)

                        extraction_audit_rows.append(
                            {
                                "run_id": int(record.run_id), "run_code": str(record.run_code),
                                "source_year": int(record.source_year), "family_key": str(record.family_key),
                                "transition_direction": str(record.transition_direction),
                                "measure": str(record.measure), "unit": str(record.unit),
                                "source_file": str(record.file_name), "source_member": str(record.csv_member),
                                "rows_read": int(len(frame)), "columns_read": int(len(frame.columns) - 10),
                                "encoding": encoding, "status": "PASS", "error": "",
                            }
                        )
                    except Exception as error:
                        extraction_audit_rows.append(
                            {
                                "run_id": int(record.run_id), "run_code": str(record.run_code),
                                "source_year": int(record.source_year), "family_key": str(record.family_key),
                                "transition_direction": str(record.transition_direction),
                                "measure": str(record.measure), "unit": str(record.unit),
                                "source_file": str(record.file_name), "source_member": str(record.csv_member),
                                "rows_read": 0, "columns_read": 0, "encoding": "", "status": "FAIL",
                                "error": f"{type(error).__name__}: {error}",
                            }
                        )

        failed_so_far = [row for row in extraction_audit_rows if row["status"] == "FAIL"]
        if failed_so_far:
            raise RuntimeError(f"Extraction failed. First failure: {failed_so_far[0]}")
        print(f"Run {run_id:02d}: read 36 files; temporary archive removed.")

extraction_audit = pd.DataFrame(extraction_audit_rows)
raw_all = pd.concat(raw_parts, ignore_index=True, sort=False)

assert len(extraction_audit) == EXPECTED_FILE_COUNT
assert extraction_audit["status"].eq("PASS").all()
assert extraction_audit["rows_read"].gt(0).all(), "At least one selected CSV was empty."

print(f"Extracted {len(raw_all):,} source rows from {len(extraction_audit)} CSV files.")
display(extraction_audit.groupby(["transition_direction", "measure"], as_index=False)["rows_read"].sum())

In [ ]:
# Step 7 - Standardize without discarding source data

def normalise_column_name(name: object) -> str:
    # Convert a source heading to a reproducible snake-case heading.
    text = str(name).strip().lower()
    text = re.sub(r"[^a-z0-9]+", "_", text).strip("_")
    return text or "unnamed"


def make_unique_columns(columns) -> list[str]:
    # Normalise headings and suffix any accidental duplicates.
    counts: dict[str, int] = {}
    result = []
    for column in columns:
        base = normalise_column_name(column)
        counts[base] = counts.get(base, 0) + 1
        result.append(base if counts[base] == 1 else f"{base}_{counts[base]}")
    return result


def first_existing(columns, exact=(), contains=()):
    # Find a preferred source heading without assuming one fixed schema.
    column_list = list(columns)
    for candidate in exact:
        if candidate in column_list:
            return candidate
    for column in column_list:
        if all(token in column for token in contains):
            return column
    return None


standardised_all = raw_all.copy()
standardised_all.columns = make_unique_columns(standardised_all.columns)
provenance_columns = {
    "run_id", "run_code", "family_key", "transition_direction", "measure", "unit",
    "source_year", "source_file", "source_member", "source_row",
}

source_columns = [column for column in standardised_all.columns if column not in provenance_columns]
year_column = first_existing(source_columns, exact=("year",), contains=("year",))
if year_column is None:
    raise KeyError("No source year column was found in the transition CSVs.")

# Canonicalise geography fields only when they exist in the source.
geography_aliases = {
    "region": first_existing(source_columns, exact=("region", "geography")),
    "region_level": first_existing(
        source_columns, exact=("region_level", "geography_level", "spatial_level"),
        contains=("region", "level"),
    ),
    "region_state": first_existing(
        source_columns, exact=("region_state", "state_region"), contains=("region", "state"),
    ),
}
rename_map = {year_column: "year"}
for canonical, found in geography_aliases.items():
    if found is not None and found != canonical:
        rename_map[found] = canonical
standardised_all = standardised_all.rename(columns=rename_map)
standardised_all["year"] = pd.to_numeric(standardised_all["year"], errors="coerce").astype("Int64")

# Detect and coalesce the measure column separately for Area and Cost.
standardised_all["value_native"] = np.nan
standardised_all["value_column_used"] = pd.Series(pd.NA, index=standardised_all.index, dtype="string")
value_candidate_columns: set[str] = set()
value_column_audit_rows = []

exact_preferences = {
    "Area": ("area_ha", "area", "value_ha", "total_area_ha", "total_area", "value"),
    "Cost": ("cost_aud", "cost", "value_aud", "total_cost_aud", "total_cost", "value"),
}
keyword_preferences = {"Area": ("area", "hectare", "ha", "value"), "Cost": ("cost", "aud", "dollar", "value")}
excluded_value_columns = provenance_columns | {"year", "region", "region_level", "region_state"}

for measure in ("Area", "Cost"):
    measure_mask = standardised_all["measure"].eq(measure)
    available_columns = [
        column for column in standardised_all.columns
        if column not in excluded_value_columns | {"value_native", "value_column_used"}
        and standardised_all.loc[measure_mask, column].notna().any()
    ]
    ordered_candidates = []
    for preferred in exact_preferences[measure]:
        if preferred in available_columns and preferred not in ordered_candidates:
            ordered_candidates.append(preferred)
    for column in available_columns:
        if any(token in column for token in keyword_preferences[measure]) and column not in ordered_candidates:
            ordered_candidates.append(column)

    numeric_candidates = {}
    for column in ordered_candidates:
        numeric = pd.to_numeric(standardised_all.loc[measure_mask, column], errors="coerce")
        if numeric.notna().any():
            numeric_candidates[column] = numeric
            value_candidate_columns.add(column)
    if not numeric_candidates:
        raise KeyError(f"No numeric {measure.lower()} value column was found. Available columns: {available_columns}")

    remaining = measure_mask.copy()
    for column, numeric in numeric_candidates.items():
        use_mask = remaining & numeric.reindex(standardised_all.index).notna()
        standardised_all.loc[use_mask, "value_native"] = numeric.reindex(standardised_all.index).loc[use_mask]
        standardised_all.loc[use_mask, "value_column_used"] = column
        remaining &= ~use_mask
    if remaining.any():
        examples = standardised_all.loc[remaining, ["source_file", "source_row"]].head(10).to_dict("records")
        raise ValueError(f"{int(remaining.sum())} {measure.lower()} rows have no numeric value. Examples: {examples}")

standardised_all["million_unit"] = np.where(standardised_all["measure"].eq("Area"), "million ha", "million AUD")
standardised_all["value_million_units"] = standardised_all["value_native"] / 1_000_000

for keys, group in standardised_all.groupby(
    ["run_id", "source_year", "family_key", "source_file", "measure"], sort=True
):
    run_id, source_year, family_key, source_file, measure = keys
    value_column_audit_rows.append({
        "run_id": int(run_id), "source_year": int(source_year), "family_key": family_key,
        "source_file": source_file, "measure": measure,
        "value_columns_used": "; ".join(sorted(group["value_column_used"].dropna().unique())),
        "rows": int(len(group)), "missing_values": int(group["value_native"].isna().sum()),
    })
value_column_audit = pd.DataFrame(value_column_audit_rows)

year_mismatch = standardised_all["year"].notna() & standardised_all["year"].ne(standardised_all["source_year"])
assert not year_mismatch.any(), f"Found {int(year_mismatch.sum())} source-year mismatches."
assert standardised_all["year"].notna().all(), "At least one source data year is missing or non-numeric."
assert standardised_all["value_native"].notna().all(), "At least one transition value is non-numeric or missing."
area_mask = standardised_all["measure"].eq("Area")
assert standardised_all.loc[area_mask, "value_native"].ge(0).all(), "At least one transition area is negative."

print("Detected canonical geography fields:", [c for c in ("region", "region_level", "region_state") if c in standardised_all])
display(value_column_audit.head(12))
display(standardised_all.head())

In [ ]:
# Step 8 - Build double count safe summaries

def clean_label(series: pd.Series) -> pd.Series:
    # Normalise labels only for matching; original labels remain unchanged.
    return series.astype("string").str.strip().str.upper().str.replace(r"[^A-Z0-9]+", "", regex=True)


def filter_required_label(frame, column, preferred_labels):
    # Select a required national label when the column is applicable.
    if column not in frame.columns or not frame[column].notna().any():
        return frame, "COLUMN_NOT_PRESENT"
    normalised = clean_label(frame[column])
    available = set(normalised.dropna().tolist())
    for label in preferred_labels:
        target = re.sub(r"[^A-Z0-9]+", "", label.upper())
        if target in available:
            return frame.loc[normalised.eq(target)].copy(), f"SELECTED:{label}"
    examples = sorted(frame[column].dropna().astype(str).unique().tolist())[:15]
    raise ValueError(f"Required national label not found in '{column}'. Example labels: {examples}")


def filter_all_or_sum_details(frame, column):
    # Prefer an explicit ALL/TOTAL row to prevent double counting.
    if column not in frame.columns or not frame[column].notna().any():
        return frame, "COLUMN_NOT_PRESENT"
    normalised = clean_label(frame[column])
    available = set(normalised.dropna().tolist())
    for label in ("ALL", "TOTAL"):
        if label in available:
            return frame.loc[normalised.eq(label)].copy(), f"SELECTED:{label}"
    return frame, "NO_ALL_LABEL_SUMMED_DETAILS"


non_dimension_columns = {
    "run_id", "run_code", "family_key", "transition_direction", "measure", "unit",
    "source_year", "source_file", "source_member", "source_row", "year",
    "value_native", "value_million_units", "million_unit", "value_column_used",
    "region", "region_level", "region_state",
} | value_candidate_columns

# Only text/category source fields can contain aggregate labels such as ALL.
categorical_dimensions = [
    column for column in standardised_all.columns
    if column not in non_dimension_columns
    and (pd.api.types.is_object_dtype(standardised_all[column])
         or pd.api.types.is_string_dtype(standardised_all[column])
         or isinstance(standardised_all[column].dtype, pd.CategoricalDtype))
]

national_by_type_rows = []
aggregation_filter_rows = []
national_detail_parts = []

group_keys = ["run_id", "run_code", "source_year", "family_key", "transition_direction", "measure", "unit"]
for keys, group in standardised_all.groupby(group_keys, dropna=False, sort=True):
    run_id, run_code, year, family_key, direction, measure, unit = keys
    working = group.copy()
    audit = {
        "run_id": int(run_id), "run_code": run_code, "year": int(year), "family_key": family_key,
        "transition_direction": direction, "measure": measure, "unit": unit,
        "source_rows": int(len(group)),
    }

    working, audit["region_rule"] = filter_required_label(working, "region", ("Australia", "AUS"))
    working, audit["region_level_rule"] = filter_required_label(working, "region_level", ("region_state",))
    working, audit["region_state_rule"] = filter_all_or_sum_details(working, "region_state")

    # Preserve detail after geography selection and before ALL/TOTAL selection.
    detail_copy = working.copy()
    detail_copy["summary_geography"] = "Australia / region_state where available"
    national_detail_parts.append(detail_copy)

    rules = []
    for dimension in categorical_dimensions:
        if dimension in working.columns and working[dimension].notna().any():
            working, rule = filter_all_or_sum_details(working, dimension)
            rules.append(f"{dimension}={rule}")

    audit["dimension_rules"] = "; ".join(rules) if rules else "NO_APPLICABLE_CATEGORICAL_DIMENSIONS"
    audit["rows_used_for_total"] = int(len(working))
    audit["value_native"] = float(working["value_native"].sum())
    aggregation_filter_rows.append(audit)
    national_by_type_rows.append({
        "run_id": int(run_id), "run_code": run_code, "year": int(year), "family_key": family_key,
        "transition_direction": direction, "measure": measure, "unit": unit,
        "value_native": float(working["value_native"].sum()),
        "value_million_units": float(working["value_native"].sum()) / 1_000_000,
        "million_unit": "million ha" if measure == "Area" else "million AUD",
        "source_rows": int(len(group)), "rows_used_for_total": int(len(working)),
    })

national_by_type = pd.DataFrame(national_by_type_rows).sort_values(
    ["run_id", "year", "family_key"]
).reset_index(drop=True)
aggregation_filters = pd.DataFrame(aggregation_filter_rows).sort_values(
    ["run_id", "year", "family_key"]
).reset_index(drop=True)
national_detail = pd.concat(national_detail_parts, ignore_index=True, sort=False)

run_year_wide = national_by_type.pivot(
    index=["run_id", "run_code", "year"], columns="family_key", values="value_native"
).reset_index()
run_year_wide.columns.name = None

print("National direction/measure summary rows:", len(national_by_type))
display(national_by_type.head(12))

In [ ]:
# Step 9 - Create summaries

def summarise_family_detail(frame: pd.DataFrame, family_key: str) -> pd.DataFrame:
    # Summarise one family at every available categorical source dimension.
    subset = frame.loc[frame["family_key"].eq(family_key)].copy()
    available_dimensions = [
        column for column in categorical_dimensions
        if column in subset.columns and subset[column].notna().any()
    ]
    group_columns = [
        "run_id", "run_code", "source_year", "family_key", "transition_direction", "measure", "unit"
    ] + available_dimensions
    summary = subset.groupby(group_columns, dropna=False, as_index=False)["value_native"].sum()
    summary = summary.rename(columns={"source_year": "year"})
    summary["value_million_units"] = summary["value_native"] / 1_000_000
    summary["million_unit"] = np.where(summary["measure"].eq("Area"), "million ha", "million AUD")
    return summary.sort_values(["run_id", "year"] + available_dimensions).reset_index(drop=True)


detail_summaries = {
    specification["sheet_prefix"] + "_detail": summarise_family_detail(national_detail, family_key)
    for family_key, specification in FAMILY_SPECS.items()
}

print("Detail summary rows by sheet:")
for sheet_name, frame in detail_summaries.items():
    print(f"  {sheet_name}: {len(frame):,}")

In [ ]:
# Step 10 - Run completeness, value and duplication check

validation_rows = []

def add_check(check, passed, observed, expected, note=""):
    validation_rows.append({
        "check": check, "status": "PASS" if bool(passed) else "FAIL",
        "observed": observed, "expected": expected, "note": note,
    })

add_check("Selected inventory file count", len(selected_inventory) == EXPECTED_FILE_COUNT, len(selected_inventory), EXPECTED_FILE_COUNT)
add_check("Extracted CSV count", len(extraction_audit) == EXPECTED_FILE_COUNT, len(extraction_audit), EXPECTED_FILE_COUNT)
add_check("All CSV extractions passed", extraction_audit["status"].eq("PASS").all(), extraction_audit["status"].value_counts().to_dict(), f"{EXPECTED_FILE_COUNT} PASS")
add_check("All selected CSVs contain rows", extraction_audit["rows_read"].gt(0).all(), int(extraction_audit["rows_read"].gt(0).sum()), EXPECTED_FILE_COUNT)
add_check("All values are numeric", standardised_all["value_native"].notna().all(), int(standardised_all["value_native"].isna().sum()), 0, "Observed is missing/non-numeric count")
area_values = standardised_all.loc[standardised_all["measure"].eq("Area"), "value_native"]
add_check("All area values are non-negative", area_values.ge(0).all(), int(area_values.lt(0).sum()), 0, "Costs are intentionally allowed to be signed")
add_check("Source year equals data year", standardised_all["year"].eq(standardised_all["source_year"]).all(), int(standardised_all["year"].ne(standardised_all["source_year"]).sum()), 0)
add_check("National summary row count", len(national_by_type) == EXPECTED_NATIONAL_ROWS, len(national_by_type), EXPECTED_NATIONAL_ROWS)
add_check("Run/year wide row count", len(run_year_wide) == EXPECTED_RUN_YEAR_ROWS, len(run_year_wide), EXPECTED_RUN_YEAR_ROWS)
add_check("Unique national summary keys", not national_by_type.duplicated(["run_id", "year", "family_key"]).any(), int(national_by_type.duplicated(["run_id", "year", "family_key"]).sum()), 0)
add_check("National values are finite", np.isfinite(national_by_type["value_native"]).all(), int((~np.isfinite(national_by_type["value_native"])).sum()), 0)

observed_years = sorted(national_by_type["year"].unique().tolist())
observed_runs = sorted(national_by_type["run_id"].unique().tolist())
observed_directions = sorted(national_by_type["transition_direction"].unique().tolist())
observed_measures = sorted(national_by_type["measure"].unique().tolist())
expected_directions = sorted({spec["transition_direction"] for spec in FAMILY_SPECS.values()})
add_check("All requested years present", observed_years == EXPECTED_YEARS, observed_years, EXPECTED_YEARS)
add_check("All 16 runs present", observed_runs == SELECTED_RUNS, observed_runs, SELECTED_RUNS)
add_check("Exactly three transition directions", observed_directions == expected_directions, observed_directions, expected_directions)
add_check("Exactly two separate measures", observed_measures == ["Area", "Cost"], observed_measures, ["Area", "Cost"])
add_check("Exactly six transition families", set(national_by_type["family_key"]) == set(FAMILY_SPECS), sorted(national_by_type["family_key"].unique()), sorted(FAMILY_SPECS))

validation_df = pd.DataFrame(validation_rows)
display(validation_df)
assert validation_df["status"].eq("PASS").all(), "A validation failed; inspect validation_df above."

In [ ]:
# Step 11 - Define Excel export helpers

def write_frame_chunks(writer, frame, sheet_prefix, manifest_rows):
    # Write every row, splitting at Excel's row limit when necessary.
    frame = frame.reset_index(drop=True)
    if frame.empty:
        sheet_name = sheet_prefix[:31]
        frame.to_excel(writer, sheet_name=sheet_name, index=False)
        manifest_rows.append({"family_key": sheet_prefix, "sheet_name": sheet_name, "first_data_row": 0, "last_data_row": 0, "rows": 0})
        return
    for chunk_number, start in enumerate(range(0, len(frame), EXCEL_DATA_ROW_LIMIT), start=1):
        stop = min(start + EXCEL_DATA_ROW_LIMIT, len(frame))
        sheet_name = f"{sheet_prefix}_raw_{chunk_number}"[:31]
        frame.iloc[start:stop].to_excel(writer, sheet_name=sheet_name, index=False)
        manifest_rows.append({
            "family_key": sheet_prefix, "sheet_name": sheet_name,
            "first_data_row": start + 1, "last_data_row": stop, "rows": stop - start,
        })


def style_workbook(path: Path):
    # Apply readable headers, filters, freeze panes and semantic number formats.
    workbook = load_workbook(path)
    header_fill = PatternFill("solid", fgColor="1F4E78")
    header_font = Font(color="FFFFFF", bold=True)

    for worksheet in workbook.worksheets:
        worksheet.sheet_view.showGridLines = False
        worksheet.freeze_panes = "A2"
        if worksheet.max_row >= 1 and worksheet.max_column >= 1:
            worksheet.auto_filter.ref = worksheet.dimensions
        headers = {}
        for cell in worksheet[1]:
            cell.fill = header_fill
            cell.font = header_font
            cell.alignment = Alignment(horizontal="center", vertical="center", wrap_text=True)
            headers[cell.column] = str(cell.value) if cell.value is not None else ""

        sample_stop = min(worksheet.max_row, 501)
        for column_number in range(1, worksheet.max_column + 1):
            values = [worksheet.cell(row=row, column=column_number).value for row in range(1, sample_stop + 1)]
            width = min(max((len(str(value)) for value in values if value is not None), default=8) + 2, 45)
            worksheet.column_dimensions[get_column_letter(column_number)].width = max(width, 10)

            header = headers.get(column_number, "").lower()
            if header in {"value_native", "area_ha", "cost_aud", "area", "cost"}:
                number_format = '#,##0.00;[Red]-#,##0.00'
            elif header in {"value_million_units"}:
                number_format = '0.000;[Red]-0.000'
            else:
                number_format = None
            if number_format:
                for cells in worksheet.iter_cols(min_col=column_number, max_col=column_number, min_row=2, max_row=worksheet.max_row):
                    for item in cells:
                        item.number_format = number_format

    workbook.save(path)

In [ ]:
# Step 12 - Write the final excel workbook

readme_rows = [
    ("Workbook", "transition-data.xlsx"),
    ("Purpose", "Extract and summarise six LUTO2 transition CSV families for Runs 1–16 and 2025–2050."),
    ("Source archive", str(RAW_ZIP_PATH)),
    ("Inventory", str(FILE_INVENTORY_PATH)),
    ("Runs", "1–16"),
    ("Years", ", ".join(map(str, EXPECTED_YEARS))),
    ("Files", "576 = 16 runs × 6 years × 3 transition directions × 2 measures"),
    ("Raw-data rule", "Every extracted source row is retained in family-specific raw sheets; see Raw_sheet_manifest."),
    ("National geography rule", "Select region=Australia and region_level=region_state when those columns exist."),
    ("Other-dimension rule", "Use ALL/TOTAL when present; otherwise sum detail rows. See Aggregation_filters."),
    ("Units", "Area is retained in hectares and cost in AUD; million-unit convenience columns are included."),
    ("Signed-cost rule", "Negative cost values are retained as reported; only area is required to be non-negative."),
    ("Mixed-unit warning", "Area and cost are different measures and must never be added together."),
    ("Direction warning", "No grand total across transition directions is calculated; use each direction separately."),
]
readme_df = pd.DataFrame(readme_rows, columns=["item", "description"])

inventory_export_columns = [column for column in [
    "run_id", "run_code", "source_year", "family_key", "transition_direction", "measure", "unit",
    "file_type", "file_name", "file_location", "file_size_mb", "compressed_size_mb",
    "source_container", "inner_archive_member", "csv_member",
] if column in selected_inventory.columns]
inventory_export = selected_inventory[inventory_export_columns].copy()

raw_sheet_manifest_rows = []
with pd.ExcelWriter(EXCEL_PATH, engine="openpyxl") as writer:
    readme_df.to_excel(writer, sheet_name="README", index=False)
    inventory_export.to_excel(writer, sheet_name="Inventory_selection", index=False)
    extraction_audit.to_excel(writer, sheet_name="Extraction_audit", index=False)
    value_column_audit.to_excel(writer, sheet_name="Value_column_audit", index=False)

    for family_key, specification in FAMILY_SPECS.items():
        family_raw = standardised_all.loc[standardised_all["family_key"].eq(family_key)]
        write_frame_chunks(writer, family_raw, specification["sheet_prefix"], raw_sheet_manifest_rows)

    national_by_type.to_excel(writer, sheet_name="National_by_type", index=False)
    run_year_wide.to_excel(writer, sheet_name="Run_year_wide", index=False)
    for sheet_name, frame in detail_summaries.items():
        frame.to_excel(writer, sheet_name=sheet_name[:31], index=False)
    aggregation_filters.to_excel(writer, sheet_name="Aggregation_filters", index=False)
    validation_df.to_excel(writer, sheet_name="Validation", index=False)

    raw_sheet_manifest = pd.DataFrame(raw_sheet_manifest_rows)
    raw_sheet_manifest.to_excel(writer, sheet_name="Raw_sheet_manifest", index=False)

style_workbook(EXCEL_PATH)
validation_df.to_csv(VALIDATION_CSV_PATH, index=False)

print("Workbook written:", EXCEL_PATH)
print("Validation CSV written:", VALIDATION_CSV_PATH)
print("Workbook size (MB):", round(EXCEL_PATH.stat().st_size / 1024**2, 2))

In [ ]:
# Step 13 - Validate saved result

saved_workbook = load_workbook(EXCEL_PATH, read_only=True, data_only=True)
saved_sheet_names = saved_workbook.sheetnames

required_sheets = {
    "README", "Inventory_selection", "Extraction_audit", "Value_column_audit",
    "National_by_type", "Run_year_wide", "Aggregation_filters", "Validation", "Raw_sheet_manifest",
} | {name[:31] for name in detail_summaries}
missing_sheets = sorted(required_sheets.difference(saved_sheet_names))
assert not missing_sheets, f"Required Excel sheets are missing: {missing_sheets}"

saved_raw_rows = sum(
    max(saved_workbook[row.sheet_name].max_row - 1, 0)
    for row in raw_sheet_manifest.itertuples(index=False)
)
assert saved_raw_rows == len(standardised_all), (
    f"Saved raw row count {saved_raw_rows:,} differs from extracted row count {len(standardised_all):,}."
)
saved_workbook.close()

saved_by_type = pd.read_excel(EXCEL_PATH, sheet_name="National_by_type")
saved_wide = pd.read_excel(EXCEL_PATH, sheet_name="Run_year_wide")
assert len(saved_by_type) == len(national_by_type) == EXPECTED_NATIONAL_ROWS
assert len(saved_wide) == len(run_year_wide) == EXPECTED_RUN_YEAR_ROWS
assert np.allclose(saved_by_type["value_native"], national_by_type["value_native"], atol=0.01, rtol=0)

roundtrip_validation = pd.DataFrame([
    {"check": "Required sheets present", "status": "PASS", "observed": len(saved_sheet_names), "expected": f">= {len(required_sheets)}"},
    {"check": "Raw rows saved", "status": "PASS", "observed": saved_raw_rows, "expected": len(standardised_all)},
    {"check": "National summary rows saved", "status": "PASS", "observed": len(saved_by_type), "expected": EXPECTED_NATIONAL_ROWS},
    {"check": "Run/year wide rows saved", "status": "PASS", "observed": len(saved_wide), "expected": EXPECTED_RUN_YEAR_ROWS},
    {"check": "Saved summary values match memory", "status": "PASS", "observed": "0.01 native-unit precision", "expected": "0.01 native-unit precision"},
])
display(roundtrip_validation)

print("All extraction, completeness, value and Excel round-trip validations passed.")
print("Final Excel file:", EXCEL_PATH)